# Notion Agent CLI vs MCP: Token Efficiency Benchmark

Controlled comparison of Notion Agent CLI (task-level actions, compact markdown-shaped I/O) vs the official Notion MCP tool path across 10 task scenarios.

In [1]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
COMPARE_RUNS = {
    "Notion Agent CLI (Sonnet)":  ("20260328-135503", "nac"),
    "Notion MCP (Sonnet)":       ("20260328-135503", "mcp"),
    "Notion Agent CLI (Opus)":   ("20260328-182242", "nac"),
    "Notion MCP (Opus)":         ("20260328-182242", "mcp"),
}
# Set COMPARE_RUNS = None to use auto-detect (picks latest nac + mcp dirs)
SELECTED_RUNS = "latest"
MIN_TURNS = 2

# ── Imports ────────────────────────────────────────────────────────────────────
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Shared chart config ───────────────────────────────────────────────────────
PALETTE = {
    "Notion Agent CLI (Sonnet)": "#2563EB", "Notion MCP (Sonnet)": "#DC2626",
    "Notion Agent CLI (Opus)":   "#7C3AED", "Notion MCP (Opus)":   "#EA580C",
    "nac": "#2563EB", "mcp": "#DC2626",
}
NAC_COLOR = "#2563EB"
MCP_COLOR = "#DC2626"
SCENARIO_NAMES = {
    1: "Read+Summarize", 2: "Query+Report", 3: "Copy+Modify", 4: "Set Property",
    5: "Replace Section", 6: "Merge Pages", 7: "Batch Update", 8: "Copy+Append",
    9: "Create Table", 10: "Import DB",
}
SHORT_NAMES = {1: "S1", 2: "S2", 3: "S3", 4: "S4", 5: "S5", 6: "S6", 7: "S7", 8: "S8", 9: "S9", 10: "S10"}
LAYOUT = dict(template="plotly_white", font_size=13)
LEGEND_TOP = dict(orientation="h", yanchor="bottom", y=1.08, x=0.5, xanchor="center")

# Resolve paths: works whether cwd is benchmark/ or project root
RESULTS_BASE = Path("results") if Path("results").exists() else (
    Path("../benchmark/results") if Path("../benchmark/results").exists() else Path("benchmark/results"))
ASSETS_DIR = RESULTS_BASE.parent / "assets" if (RESULTS_BASE.parent / "assets").exists() else (
    Path("benchmark/assets") if Path("benchmark/assets").exists() else RESULTS_BASE.parent / "assets")

# ── Stats helpers ──────────────────────────────────────────────────────────────
def ci_95(values):
    """Return (lo, hi) 95% CI using t-distribution. Returns (nan, nan) if n < 2."""
    n = len(values)
    if n < 2:
        return (np.nan, np.nan)
    m = np.mean(values)
    se = stats.sem(values)
    lo, hi = stats.t.interval(0.95, df=n - 1, loc=m, scale=se)
    return (lo, hi)

def cohens_d(a, b):
    """Cohen's d for independent samples. Returns nan if either group has < 2 values."""
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled_std = np.sqrt(((len(a) - 1) * a.std(ddof=1)**2 + (len(b) - 1) * b.std(ddof=1)**2) / (len(a) + len(b) - 2))
    if pooled_std == 0:
        return 0.0
    return (a.mean() - b.mean()) / pooled_std

In [2]:
# ── Environment metadata ──────────────────────────────────────────────────────
if COMPARE_RUNS:
    _rids = list({v[0] if isinstance(v, (list, tuple)) else v for v in COMPARE_RUNS.values()})
else:
    _rids = SELECTED_RUNS if isinstance(SELECTED_RUNS, list) else []

for rid in _rids:
    p = RESULTS_BASE / rid / "env.json"
    if p.exists():
        e = json.loads(p.read_text())
        print(f"{rid}: model={e.get('model')}, cli={e.get('claude_cli_version')}")

20260328-182242: model=claude-opus-4-6, cli=2.1.86 (Claude Code)
20260328-135503: model=claude-sonnet-4-6, cli=2.1.86 (Claude Code)


## 1. Data Loading

In [3]:
def load_from_summary(run_id: str, label_override=None, file_prefix=None) -> pd.DataFrame:
    """Primary path: load sessions from summary.json (generated by run.py)."""
    summary_path = RESULTS_BASE / run_id / "summary.json"
    if not summary_path.exists():
        return pd.DataFrame()
    data = json.loads(summary_path.read_text())
    sessions = []
    for s in data.get("sessions", []):
        condition = s.get("condition", "")
        if file_prefix and condition != file_prefix:
            continue
        if s.get("num_turns", 0) < MIN_TURNS:
            continue
        snum = s.get("scenario_num", 0)
        sessions.append({
            "run_id": run_id,
            "name": s["label"],
            "plugin": condition,
            "label": label_override or condition,
            "scenario": f"S{snum} {SCENARIO_NAMES.get(snum, '')}",
            "scenario_short": SHORT_NAMES.get(snum, f"S{snum}"),
            "scenario_num": snum,
            "iteration": s.get("iteration", 0),
            "num_turns": s["num_turns"],
            "cost_usd": s.get("cost_usd", 0),
            "duration_s": s.get("duration_ms", 0) / 1000,
            "total_input": s.get("input_tokens", 0),
            "total_output": s.get("output_tokens", 0),
            "total_cache_read": s.get("cache_read_tokens", 0),
            "total_cache_creation": s.get("cache_creation_tokens", 0),
            "billed_input": s.get("input_tokens", 0) + s.get("cache_creation_tokens", 0),
            "turn_tokens": s.get("turn_tokens", []),
            "valid": not s.get("contaminated", False) and s.get("valid", True) is not False,
            "validated": s.get("valid"),
            "validation_reason": s.get("validation_reason"),
            "workflow_match": s.get("workflow_match"),
            "first_useful_turn": s.get("first_useful_turn"),
            "action_sequence": s.get("action_sequence", []),
            "intended_workflow": s.get("intended_workflow", []),
            "mismatch_reason": s.get("mismatch_reason"),
            "schema_used": s.get("schema_used", False),
            "source_inspection": s.get("source_inspection", False),
        })
    return pd.DataFrame(sessions)


# ── Fallback: raw file parsing (for runs without summary.json) ─────────────────
def parse_summary(path: Path) -> dict | None:
    try:
        data = json.loads(path.read_text())
    except (json.JSONDecodeError, OSError):
        return None
    result = str(data.get("result", ""))
    return {
        "num_turns": data.get("num_turns", 0),
        "cost_usd": data.get("total_cost_usd", data.get("cost_usd", 0)),
        "is_error": any(s in result for s in ["hit your limit", "rate_limit"]),
        "duration_ms": data.get("total_duration_ms", 0),
    }

def parse_jsonl(path: Path) -> dict:
    try:
        lines = [json.loads(l) for l in path.read_text().strip().splitlines() if l.strip()]
    except (json.JSONDecodeError, OSError):
        return {}
    if lines and lines[0].get("error") == "jsonl_missing":
        return {"tools_used": "missing"}
    total_input = total_output = total_cache_read = total_cache_creation = 0
    has_bash = has_mcp = False
    prev_sig = None
    turns = []
    for r in lines:
        if r.get("type") != "assistant":
            continue
        usage = r.get("message", {}).get("usage", {})
        if not usage:
            continue
        inp = usage.get("input_tokens", 0)
        cr = usage.get("cache_read_input_tokens", 0)
        cc = usage.get("cache_creation_input_tokens", 0)
        out = usage.get("output_tokens", 0)
        sig = (inp, cr, cc)
        is_new = sig != prev_sig
        prev_sig = sig
        if is_new:
            total_input += inp; total_cache_read += cr; total_cache_creation += cc
            turns.append({"input": inp, "cache_read": cr, "cache_create": cc,
                          "output": out, "billed": inp + cc})
        else:
            if turns:
                turns[-1]["output"] += out
        total_output += out
        for item in r.get("message", {}).get("content", []):
            if isinstance(item, dict) and item.get("type") == "tool_use":
                name = item.get("name", "")
                if name == "Bash": has_bash = True
                elif "__" in name: has_mcp = True
    tools_used = ("mixed" if has_bash and has_mcp else "cli" if has_bash else "mcp" if has_mcp else "none")
    return {"total_input": total_input, "total_output": total_output,
            "total_cache_read": total_cache_read, "total_cache_creation": total_cache_creation,
            "billed_input": total_input + total_cache_creation, "tools_used": tools_used,
            "turn_tokens": turns}

def load_run_fallback(run_id: str, label_override=None, file_prefix=None):
    """Fallback for runs without summary.json."""
    run_dir = RESULTS_BASE / run_id
    sessions = []
    for f in sorted(run_dir.glob("*.json")):
        if f.name in ("env.json", "behavior.json", "validation.json", "summary.json"):
            continue
        if file_prefix and not f.stem.startswith(file_prefix + "-"):
            continue
        parts = f.stem.split("-")
        if len(parts) < 3:
            continue
        s = parse_summary(f)
        if not s or s["is_error"] or s["num_turns"] < MIN_TURNS:
            continue
        jd = parse_jsonl(f.with_suffix(".jsonl")) if f.with_suffix(".jsonl").exists() else {}
        plugin = parts[0]
        tu = jd.get("tools_used", "unknown")
        valid = tu not in ("mixed", "missing")
        if plugin == "mcp" and tu == "cli": valid = False
        if plugin == "nac" and tu == "mcp": valid = False
        snum = int(parts[1][1:])
        sessions.append({"run_id": run_id, "name": f.stem, "plugin": plugin,
            "label": label_override or plugin,
            "scenario": f"S{snum} {SCENARIO_NAMES.get(snum, '')}",
            "scenario_short": SHORT_NAMES.get(snum, f"S{snum}"),
            "scenario_num": snum, "iteration": int(parts[2]),
            "num_turns": s["num_turns"], "cost_usd": s["cost_usd"],
            "duration_s": s["duration_ms"] / 1000,
            "tools_used": tu, "valid": valid,
            **{k: v for k, v in jd.items() if k not in ("tools_used", "turn_tokens")},
            "turn_tokens": jd.get("turn_tokens", [])})
    return pd.DataFrame(sessions)


def load_run(run_id: str, label_override=None, file_prefix=None):
    """Load from summary.json if available, otherwise fall back to raw parsing."""
    summary_path = RESULTS_BASE / run_id / "summary.json"
    if summary_path.exists():
        return load_from_summary(run_id, label_override, file_prefix)
    return load_run_fallback(run_id, label_override, file_prefix)


# ── Build DataFrame ────────────────────────────────────────────────────────────
if COMPARE_RUNS:
    dfs = []
    for label, val in COMPARE_RUNS.items():
        rid, prefix = (val if isinstance(val, (list, tuple)) else (val, None))
        rdf = load_run(rid, label_override=label, file_prefix=prefix)
        if not rdf.empty: dfs.append(rdf)
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    labels = list(COMPARE_RUNS.keys())
else:
    all_runs = [d.name for d in sorted(RESULTS_BASE.iterdir()) if d.is_dir() and not d.name.startswith("_")]
    if SELECTED_RUNS == "latest":
        selected, has_nac, has_mcp = [], False, False
        for rid in reversed(all_runs):
            d = RESULTS_BASE / rid
            if any(d.glob("nac-*.json")) and not has_nac:
                has_nac = True; selected.append(rid)
            if any(d.glob("mcp-*.json")) and not has_mcp:
                has_mcp = True
                if rid not in selected: selected.append(rid)
            if has_nac and has_mcp: break
    else:
        selected = SELECTED_RUNS if isinstance(SELECTED_RUNS, list) else [SELECTED_RUNS]
    dfs = [load_run(rid) for rid in selected]
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    df["label"] = df["plugin"]
    labels = sorted(df.label.unique().tolist())

if "valid" in df.columns:
    n_bad = (~df.valid).sum()
    if n_bad: print(f"Excluded {n_bad} contaminated sessions")
    df = df[df.valid].copy()

run_label = " vs ".join(labels)
print(f"{run_label}: {len(df)} sessions, scenarios {sorted(df.scenario_num.unique())}")

Notion Agent CLI (Sonnet) vs Notion MCP (Sonnet) vs Notion Agent CLI (Opus) vs Notion MCP (Opus): 200 sessions, scenarios [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


## 2. Summary

In [4]:
# Build enhanced summary with CIs and Cohen's d
def _model_pairs(labels):
    pairs = []
    for model in ["Sonnet", "Opus"]:
        cli = [l for l in labels if "CLI" in l and model in l]
        mcp = [l for l in labels if "MCP" in l and model in l]
        if cli and mcp:
            pairs.append((model, cli[0], mcp[0]))
    if not pairs:
        cli = [l for l in labels if "nac" in l.lower() or "cli" in l.lower()]
        mcp = [l for l in labels if "mcp" in l.lower()]
        if cli and mcp:
            pairs.append(("", cli[0], mcp[0]))
    return pairs

header_vals = ["Scenario"]
pairs = _model_pairs(labels)
for model, cli_l, mcp_l in pairs:
    pfx = f"{model} " if model else ""
    header_vals += [f"{pfx}Notion Agent CLI mean (95% CI)", f"{pfx}MCP mean (95% CI)", f"{pfx}Saving %", f"{pfx}Cohen's d"]

col_data = {h: [] for h in header_vals}

for s in sorted(df.scenario_num.unique()):
    col_data["Scenario"].append(f"S{s} {SCENARIO_NAMES.get(s, '')}")
    for model, cli_l, mcp_l in pairs:
        pfx = f"{model} " if model else ""
        nac_sub = df[(df.scenario_num == s) & (df.label == cli_l)]
        mcp_sub = df[(df.scenario_num == s) & (df.label == mcp_l)]

        for sub, col_key in [(nac_sub, f"{pfx}Notion Agent CLI mean (95% CI)"), (mcp_sub, f"{pfx}MCP mean (95% CI)")]:
            if sub.empty:
                col_data[col_key].append("-")
                continue
            m = sub.cost_usd.mean()
            lo, hi = ci_95(sub.cost_usd.values)
            if np.isnan(lo):
                col_data[col_key].append(f"${m:.4f}")
            else:
                col_data[col_key].append(f"${m:.4f} (${lo:.4f}-${hi:.4f})")

        # Saving %
        if not nac_sub.empty and not mcp_sub.empty:
            nm, mm = nac_sub.cost_usd.mean(), mcp_sub.cost_usd.mean()
            saving = (1 - nm / mm) * 100 if mm > 0 else 0
            col_data[f"{pfx}Saving %"].append(f"{saving:.0f}%")
        else:
            col_data[f"{pfx}Saving %"].append("-")

        # Cohen's d
        if not nac_sub.empty and not mcp_sub.empty:
            d = cohens_d(mcp_sub.cost_usd.values, nac_sub.cost_usd.values)
            col_data[f"{pfx}Cohen's d"].append(f"{d:.2f}" if not np.isnan(d) else "-")
        else:
            col_data[f"{pfx}Cohen's d"].append("-")

# Totals
col_data["Scenario"].append("<b>TOTAL</b>")
for model, cli_l, mcp_l in pairs:
    pfx = f"{model} " if model else ""
    nac_all = df[df.label == cli_l]
    mcp_all = df[df.label == mcp_l]
    col_data[f"{pfx}Notion Agent CLI mean (95% CI)"].append(f"<b>${nac_all.cost_usd.sum():.2f}</b>")
    col_data[f"{pfx}MCP mean (95% CI)"].append(f"<b>${mcp_all.cost_usd.sum():.2f}</b>")
    nt, mt = nac_all.cost_usd.sum(), mcp_all.cost_usd.sum()
    col_data[f"{pfx}Saving %"].append(f"<b>{(1-nt/mt)*100:.0f}%</b>" if mt else "-")
    col_data[f"{pfx}Cohen's d"].append("")

fig = go.Figure(go.Table(
    header=dict(values=header_vals, fill_color="#f8fafc", font=dict(size=10),
                align=["left"] + ["right"] * (len(header_vals) - 1), line_color="#e2e8f0"),
    cells=dict(values=[col_data[h] for h in header_vals],
               font=dict(size=10), align=["left"] + ["right"] * (len(header_vals) - 1),
               line_color="#e2e8f0", height=28),
))
fig.update_layout(title=f"{run_label} — Enhanced Summary (95% CI, Cohen's d)",
                  height=60 + 28 * (len(col_data["Scenario"]) + 1),
                  margin=dict(t=40, b=10, l=10, r=10))
fig.show()

display(Markdown("*Note: n=5 per scenario per condition. CIs are reported for transparency but "
                 "n=5 is too small for formal significance testing. Cohen's d > 0.8 is conventionally large.*"))

*Note: n=5 per scenario per condition. CIs are reported for transparency but n=5 is too small for formal significance testing. Cohen's d > 0.8 is conventionally large.*

## 3. Mean Session Cost by Scenario

Horizontal bars show mean cost per session (USD). Error bars are ± 1 SD. n=5 iterations per condition per scenario. Both panels share the same x-axis for cross-model comparison.

In [5]:
charts = {}

pairs = _model_pairs(labels)
agg = df.groupby(["scenario_num", "label"]).agg(
    cost_mean=("cost_usd", "mean"), cost_std=("cost_usd", "std"),
    n=("cost_usd", "count")).reset_index()
scenario_labels = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg.scenario_num.unique())]
x_max = (agg.cost_mean + agg.cost_std.fillna(0)).max() * 1.30

fig = make_subplots(rows=1, cols=len(pairs),
    subplot_titles=["Sonnet 4.6", "Opus 4.6"][:len(pairs)],
    horizontal_spacing=0.20, shared_yaxes=True)

for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
    for label, color, name in [(mcp_label, MCP_COLOR, "MCP"), (cli_label, NAC_COLOR, "Notion Agent CLI")]:
        sub = agg[agg.label == label].sort_values("scenario_num")
        sub_labels = sub.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")
        texts = [f"${r.cost_mean:.3f}" for _, r in sub.iterrows()] if name == "MCP" else [""] * len(sub)
        fig.add_trace(go.Bar(
            y=sub_labels, x=sub.cost_mean, orientation="h",
            error_x=dict(type="data", array=sub.cost_std.fillna(0), thickness=1.5, width=3),
            name=name, legendgroup=name, showlegend=(col == 1),
            marker_color=color, opacity=0.85,
            text=texts, textposition="outside", textfont_size=10,
            hovertemplate="%{y}<br>" + name + ": $%{x:.4f} +/- $%{error_x.array:.4f}<extra></extra>",
        ), row=1, col=col)
    fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_labels)), row=1, col=col)
    fig.update_xaxes(title_text="USD", tickprefix="$", range=[0, x_max], row=1, col=col)

fig.update_layout(**LAYOUT, barmode="group", height=540, width=1100,
    title=dict(text="Mean Session Cost by Scenario (+/- SD)",
               y=0.98, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=100, b=50, l=160, r=80),
    legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.5, xanchor="center"))
charts["bench-turns-cost"] = fig
fig.show()

## 3b. Paired Comparison (Turns + Cost by Model)

In [6]:
agg_paired = df.groupby(["scenario_num", "scenario_short", "label"]).agg(
    turns_mean=("num_turns", "mean"), cost_mean=("cost_usd", "mean")).reset_index()
scenario_labels_full = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg_paired.scenario_num.unique())]

titles = [f"Turns — {m}" for m, _, _ in pairs] + [f"Cost — {m}" for m, _, _ in pairs]
fig = make_subplots(rows=2, cols=len(pairs), subplot_titles=titles,
    horizontal_spacing=0.16, vertical_spacing=0.16)

turns_max = float(agg_paired.turns_mean.max()) * 1.20
cost_max = float(agg_paired.cost_mean.max()) * 1.20

for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
    cli = agg_paired[agg_paired.label == cli_label][["scenario_num", "turns_mean", "cost_mean"]].rename(
        columns={"turns_mean": "turns_cli", "cost_mean": "cost_cli"})
    mcp = agg_paired[agg_paired.label == mcp_label][["scenario_num", "turns_mean", "cost_mean"]].rename(
        columns={"turns_mean": "turns_mcp", "cost_mean": "cost_mcp"})
    merged = cli.merge(mcp, on="scenario_num", how="inner").sort_values("scenario_num")
    merged["scenario_label"] = merged["scenario_num"].map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")

    for row, (cli_m, mcp_m, x_title, x_max, fmt) in enumerate([
        ("turns_cli", "turns_mcp", "Turns", turns_max, "%{x:.1f} turns"),
        ("cost_cli", "cost_mcp", "USD", cost_max, "$%{x:.3f}"),
    ], 1):
        for _, item in merged.iterrows():
            lo, hi = sorted((item[cli_m], item[mcp_m]))
            fig.add_trace(go.Scatter(x=[lo, hi], y=[item["scenario_label"], item["scenario_label"]],
                mode="lines", line=dict(color="#CBD5E1", width=3),
                showlegend=False, hoverinfo="skip"), row=row, col=col)
        fig.add_trace(go.Scatter(x=merged[cli_m], y=merged["scenario_label"], mode="markers",
            name="Notion Agent CLI", legendgroup="Notion Agent CLI", showlegend=(col == 1 and row == 1),
            marker=dict(color=NAC_COLOR, size=11, line=dict(color="white", width=1)),
            hovertemplate="%{y}<br>Notion Agent CLI: " + fmt + "<extra>" + model + "</extra>"),
            row=row, col=col)
        fig.add_trace(go.Scatter(x=merged[mcp_m], y=merged["scenario_label"], mode="markers",
            name="MCP", legendgroup="MCP", showlegend=(col == 1 and row == 1),
            marker=dict(color=MCP_COLOR, size=11, line=dict(color="white", width=1)),
            hovertemplate="%{y}<br>MCP: " + fmt + "<extra>" + model + "</extra>"),
            row=row, col=col)

        fig.update_xaxes(title_text=x_title, range=[0, x_max], row=row, col=col)
        if row == 2:
            fig.update_xaxes(tickprefix="$", row=row, col=col)
        fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_labels_full)),
            showticklabels=(col == 1), row=row, col=col)

fig.update_layout(**LAYOUT, height=760, width=1180,
    title=dict(text="Per-Scenario Comparison: Notion Agent CLI vs MCP by Model",
               y=0.985, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=120, b=60, l=170, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center"))
fig.show()
charts["bench-turns-cost-paired"] = fig


## 3c. Individual Sessions (raw points)

Each dot is one session. Bars show the mean. With n=5, every data point is visible, making outliers and clustering patterns transparent.

In [7]:
agg_pts = df.groupby(["scenario_num", "label"]).agg(
    cost_mean=("cost_usd", "mean")).reset_index()
scenario_labels_pts = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg_pts.scenario_num.unique())]
x_max_pts = df.cost_usd.max() * 1.15

fig = make_subplots(rows=1, cols=len(pairs),
    subplot_titles=["Sonnet 4.6", "Opus 4.6"][:len(pairs)],
    horizontal_spacing=0.20, shared_yaxes=True)

for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
    for label, color, name in [(mcp_label, MCP_COLOR, "MCP"), (cli_label, NAC_COLOR, "Notion Agent CLI")]:
        # Mean bars (thin, semi-transparent)
        sub_agg = agg_pts[agg_pts.label == label].sort_values("scenario_num")
        sub_labels = sub_agg.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")
        fig.add_trace(go.Bar(
            y=sub_labels, x=sub_agg.cost_mean, orientation="h",
            name=name + " mean", legendgroup=name, showlegend=False,
            marker_color=color, opacity=0.25, width=0.35,
            hoverinfo="skip",
        ), row=1, col=col)

        # Raw session points (jittered vertically)
        sub_raw = df[df.label == label].sort_values(["scenario_num", "iteration"])
        raw_labels = sub_raw.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")

        fig.add_trace(go.Scatter(
            x=sub_raw.cost_usd, y=raw_labels, mode="markers",
            name=name, legendgroup=name, showlegend=(col == 1),
            marker=dict(color=color, size=8, opacity=0.85,
                        line=dict(color="white", width=0.5)),
            hovertemplate="%{y}<br>" + name + ": $%{x:.4f}<br>%{text}<extra></extra>",
            text=sub_raw.name,
        ), row=1, col=col)

    fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_labels_pts)),
        row=1, col=col)
    fig.update_xaxes(title_text="USD", tickprefix="$", range=[0, x_max_pts], row=1, col=col)

fig.update_layout(**LAYOUT, barmode="overlay", height=540, width=1100,
    title=dict(text="Session Cost: Individual Observations",
               y=0.98, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=100, b=50, l=160, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.5, xanchor="center"))
fig.show()

## 4. Session Variability

In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Turns per Session", "Cost per Session"],
                    horizontal_spacing=0.14)

for label in labels:
    sub = df[df.label == label].sort_values("scenario_num")
    color = PALETTE.get(label, "#888")

    fig.add_trace(go.Box(
        x=sub.scenario_short, y=sub.num_turns, name=label, legendgroup=label,
        marker_color=color, boxpoints="all", jitter=0.4, pointpos=0,
        marker_size=6, line_width=1.5,
        hovertemplate="%{x}<br>Turns: %{y}<extra>" + label + "</extra>",
        showlegend=True,
    ), row=1, col=1)

    fig.add_trace(go.Box(
        x=sub.scenario_short, y=sub.cost_usd, name=label, legendgroup=label,
        marker_color=color, boxpoints="all", jitter=0.4, pointpos=0,
        marker_size=6, line_width=1.5,
        hovertemplate="%{x}<br>Cost: $%{y:.4f}<extra>" + label + "</extra>",
        showlegend=False,
    ), row=1, col=2)

fig.update_layout(**LAYOUT, height=520, width=1100, boxmode="group",
    title=dict(text="Session Variability", y=0.98, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=120, b=50, l=60, r=30), legend=LEGEND_TOP)
fig.update_yaxes(title_text="Turns", row=1, col=1)
fig.update_yaxes(title_text="USD", tickprefix="$", row=1, col=2)
fig.show()

## 4b. Validation Heatmap

In [9]:
# Validation heatmap: scenario x iteration, colored pass/fail
if "validated" in df.columns:
    nac_df = df[df.plugin == "nac"] if "plugin" in df.columns else df[df.label.str.contains("NAC|nac", case=False)]
    if not nac_df.empty:
        scenarios = sorted(nac_df.scenario_num.unique())
        iterations = sorted(nac_df.iteration.unique())

        z = []
        hover = []
        for s in scenarios:
            row_z = []
            row_h = []
            for it in iterations:
                sub = nac_df[(nac_df.scenario_num == s) & (nac_df.iteration == it)]
                if sub.empty:
                    row_z.append(0.5)
                    row_h.append("missing")
                else:
                    v = sub.iloc[0].get("validated")
                    row_z.append(1.0 if v else 0.0)
                    reason = sub.iloc[0].get("validation_reason", "")
                    row_h.append(f"{'PASS' if v else 'FAIL'}{': ' + reason if reason else ''}")
            z.append(row_z)
            hover.append(row_h)

        fig = go.Figure(go.Heatmap(
            z=z, x=[f"Iter {i}" for i in iterations],
            y=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in scenarios],
            text=hover, hovertemplate="%{y}, %{x}: %{text}<extra></extra>",
            colorscale=[[0, "#EF4444"], [0.5, "#D1D5DB"], [1, "#22C55E"]],
            showscale=False, zmin=0, zmax=1,
        ))
        fig.update_layout(**LAYOUT, height=400, width=600,
            title="Notion Agent CLI Validation Results (pass=green, fail=red)",
            margin=dict(t=60, b=40, l=160, r=30))
        fig.show()
    else:
        print("No Notion Agent CLI sessions with validation data")
else:
    print("No validation data available (run with summary.json)")

## 4c. Workflow Adherence

In [10]:
# Workflow adherence: match % per scenario, actual vs intended, first useful turn distribution
if "workflow_match" in df.columns:
    nac_labels = [l for l in labels if "NAC" in l or "nac" in l.lower() or "CLI" in l]
    nac_df = df[df.label.isin(nac_labels)] if nac_labels else df[df.plugin == "nac"]

    if not nac_df.empty and nac_df.workflow_match.notna().any():
        # Bar chart: match % per scenario
        wf_agg = nac_df.groupby("scenario_num").agg(
            match_pct=("workflow_match", "mean"),
            n=("workflow_match", "count"),
        ).reset_index()
        wf_agg["scenario_label"] = wf_agg.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")

        fig = go.Figure(go.Bar(
            x=wf_agg.scenario_label, y=wf_agg.match_pct * 100,
            marker_color=[NAC_COLOR if v == 1.0 else "#F59E0B" for v in wf_agg.match_pct],
            text=[f"{v:.0f}%" for v in wf_agg.match_pct * 100],
            textposition="outside",
        ))
        fig.update_layout(**LAYOUT, height=400, width=900,
            title="Notion Agent CLI Workflow Adherence by Scenario",
            yaxis_title="Match %", yaxis_range=[0, 110],
            margin=dict(t=60, b=60))
        fig.show()

        # Table: actual vs intended per scenario
        wf_rows = []
        for s in sorted(nac_df.scenario_num.unique()):
            sub = nac_df[nac_df.scenario_num == s]
            intended = sub.iloc[0].get("intended_workflow", [])
            intended_str = " -> ".join(intended) if isinstance(intended, list) else str(intended)
            actual_seqs = sub.action_sequence.apply(
                lambda x: " -> ".join(x) if isinstance(x, list) else str(x)
            ).unique()
            match_pct = sub.workflow_match.mean() * 100
            wf_rows.append({
                "Scenario": f"S{s} {SCENARIO_NAMES.get(s, '')}",
                "Intended": intended_str,
                "Actual (unique)": " | ".join(actual_seqs[:3]),
                "Match": f"{match_pct:.0f}%",
            })
        display(pd.DataFrame(wf_rows))

        # First useful turn distribution
        fut = nac_df[nac_df.first_useful_turn.notna()].first_useful_turn
        if not fut.empty:
            fig2 = go.Figure(go.Histogram(x=fut, nbinsx=int(fut.max()),
                marker_color=NAC_COLOR, opacity=0.8))
            fig2.update_layout(**LAYOUT, height=300, width=600,
                title="Distribution of First Useful Turn (Notion Agent CLI)",
                xaxis_title="Turn number", yaxis_title="Count",
                margin=dict(t=60, b=50))
            fig2.show()
    else:
        print("No workflow data available")
else:
    print("No workflow_match column (requires summary.json)")

,Scenario,Intended,Actual (unique),Match
0,S1 Read+Summarize,getPage -> createPage,getPage -> createPage,100%
1,S2 Query+Report,queryDatabase -> createPage,queryDatabase -> createPage | queryDatabase ->...,100%
2,S3 Copy+Modify,copyPageWith,getPage -> copyPageWith | copyPageWith,100%
3,S4 Set Property,setProperties,setProperties,100%
4,S5 Replace Section,replaceSection,replaceSection,100%
5,S6 Merge Pages,createPage -> mergePages,createPage -> mergePages,100%
6,S7 Batch Update,batchSetProperties,batchSetProperties,100%
7,S8 Copy+Append,copyPageWith,copyPageWith,100%
8,S9 Create Table,createPage,createPage | createPage -> createPage -> creat...,100%
9,S10 Import DB,importTable,importTable -> importTable | importTable |,90%


## 5. Token Breakdown

In [11]:
import math
token_cols = ["total_input", "total_cache_read", "total_cache_creation", "total_output"]
token_labels = {"total_input": "New input", "total_cache_read": "Cache read (75% off)",
                "total_cache_creation": "Cache creation", "total_output": "Output"}
token_colors = {"New input": "#2563EB", "Cache read (75% off)": "#10B981",
                "Cache creation": "#F59E0B", "Output": "#EF4444"}

if all(c in df.columns for c in token_cols):
    scenarios = sorted(df.scenario_num.unique())
    n = len(scenarios)
    ncols = min(5, n)
    nrows = math.ceil(n / ncols)
    fig = make_subplots(rows=nrows, cols=ncols,
        subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in scenarios],
        horizontal_spacing=0.06, vertical_spacing=0.15)

    # Short legend labels for each condition
    short_labels = {l: l.replace("Notion MCP", "MCP") for l in labels}

    for idx, s in enumerate(scenarios):
        row, col = idx // ncols + 1, idx % ncols + 1
        sub = df[df.scenario_num == s]

        for comp_col, comp_name in token_labels.items():
            for li, label in enumerate(labels):
                lsub = sub[sub.label == label]
                if lsub.empty: continue
                val = lsub[comp_col].mean()
                fig.add_trace(go.Bar(
                    name=comp_name, x=[short_labels[label]], y=[val],
                    marker_color=token_colors[comp_name],
                    showlegend=(idx == 0 and li == 0),
                    legendgroup=comp_name,
                    hovertemplate=f"{short_labels[label]}<br>{comp_name}: %{{y:,.0f}}<extra>S{s}</extra>",
                ), row=row, col=col)

    fig.update_layout(**LAYOUT, barmode="stack", height=700, width=1100,
        title=dict(text="Token Breakdown per Scenario", y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=100, b=40, l=60, r=30),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center"))
    for r in range(1, nrows + 1):
        fig.update_yaxes(title_text="Tokens", row=r, col=1)
    fig.show()
else:
    print("Token data not available")

## 6. Savings

In [12]:
pairs = _model_pairs(labels)
if pairs:
    ncols = len(pairs)
    fig = make_subplots(rows=2, cols=ncols,
        subplot_titles=[f"Turn Reduction — {m}" for m, _, _ in pairs]
                     + [f"Cost Reduction — {m}" for m, _, _ in pairs],
        horizontal_spacing=0.16, vertical_spacing=0.14, shared_yaxes=True)

    for col, (model, cli_l, mcp_l) in enumerate(pairs, 1):
        rows = []
        for s in sorted(df.scenario_num.unique()):
            sa = df[(df.scenario_num == s) & (df.label == cli_l)]
            sb = df[(df.scenario_num == s) & (df.label == mcp_l)]
            if sa.empty or sb.empty: continue
            at, bt = sa.num_turns.mean(), sb.num_turns.mean()
            ac, bc = sa.cost_usd.mean(), sb.cost_usd.mean()
            rows.append({"scenario": f"S{s} {SCENARIO_NAMES.get(s, '')}",
                "turn_saving": (1 - at / bt) * 100 if bt else 0,
                "cost_saving": (1 - ac / bc) * 100 if bc else 0})
        sdf = pd.DataFrame(rows)

        for row_idx, metric in enumerate(["turn_saving", "cost_saving"], 1):
            colors = ["#10B981" if v > 0 else "#EF4444" for v in sdf[metric]]
            fig.add_trace(go.Bar(
                y=sdf.scenario, x=sdf[metric], orientation="h",
                marker_color=colors, showlegend=False,
                text=sdf[metric].apply(lambda v: f"{v:+.0f}%"),
                textposition="outside", textfont_size=10,
            ), row=row_idx, col=col)
            fig.add_vline(x=0, line_dash="dash", line_color="#ccc", line_width=1, row=row_idx, col=col)
            fig.update_xaxes(title_text="%", range=[0, 108], row=row_idx, col=col)

        # Annotation per model
        a_t = df[df.label == cli_l].cost_usd.sum()
        b_t = df[df.label == mcp_l].cost_usd.sum()
        total_pct = (1 - a_t / b_t) * 100 if b_t else 0

    fig.update_layout(**LAYOUT, showlegend=False, height=700, width=500 * ncols + 200,
        title=dict(text="Savings: Notion Agent CLI vs MCP by Model (positive = Notion Agent CLI wins)",
                   y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=100, b=50, l=160, r=70))
    charts["bench-savings"] = fig
    fig.show()

## 7. Cumulative Cost per Turn

Shows how billed input tokens accumulate as turns progress. Each turn re-sends the full conversation context, so more turns = quadratic cost growth. NAC finishes in 2-3 turns; MCP needs 4-40.

In [13]:
# Cumulative billed tokens per turn — showcase 6 scenarios that best illustrate context growth
showcase = [1, 2, 3, 6, 8, 10]
available = [s for s in showcase if s in df.scenario_num.values]
if len(available) < 3:
    available = sorted(df.scenario_num.unique())[:6]

ncols, nrows = 3, 2
short = {l: l.replace("Notion MCP", "MCP") for l in labels}

fig = make_subplots(rows=nrows, cols=ncols,
    subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in available],
    horizontal_spacing=0.10, vertical_spacing=0.18)

for idx, s in enumerate(available):
    r, c = idx // ncols + 1, idx % ncols + 1
    for label in labels:
        sub = df[(df.scenario_num == s) & (df.label == label) & (df.iteration == 1)]
        if sub.empty:
            continue
        tt = sub.iloc[0].get("turn_tokens", [])
        if not tt:
            continue
        billed = [t.get("billed", t.get("input", 0) + t.get("cache_create", 0)) for t in tt]
        cum = np.cumsum(billed)
        color = PALETTE.get(label, "#888")
        fig.add_trace(go.Scatter(
            x=list(range(1, len(cum) + 1)), y=cum,
            mode="lines+markers", name=short[label], legendgroup=label,
            line=dict(color=color, width=2.5), marker=dict(size=5),
            showlegend=(idx == 0),
        ), row=r, col=c)
    fig.update_xaxes(dtick=1, row=r, col=c)

fig.update_yaxes(title_text="Billed tokens", row=1, col=1)
fig.update_yaxes(title_text="Billed tokens", row=2, col=1)
fig.update_layout(**LAYOUT, height=600, width=1000,
    title=dict(text="Context Growth per Turn (billed input, iteration 1)",
               y=0.98, x=0.5, xanchor="center", font_size=15),
    margin=dict(t=90, b=40, l=80, r=30),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center",
                font_size=11))
charts["bench-cumulative"] = fig
fig.show()

## 7b. Per-Turn Token Breakdown (Stacked)

In [14]:
# Per-turn stacked token breakdown — one Notion Agent CLI label, 6 scenarios, iteration 1
nac_label = next((l for l in labels if "CLI" in l and "Sonnet" in l), 
                 next((l for l in labels if "NAC" in l or "nac" in l.lower() or "CLI" in l), labels[0]))

rep_scenarios = sorted(df.scenario_num.unique())[:6]
ncols, nrows = 3, 2

fig = make_subplots(rows=nrows, cols=ncols,
    subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in rep_scenarios],
    horizontal_spacing=0.10, vertical_spacing=0.18)

categories = [
    ("input", "New Input", "#60A5FA"),
    ("cache_read", "Cache Read", "#34D399"),
    ("cache_create", "Cache Create", "#FBBF24"),
    ("output", "Output", "#F87171"),
]
legend_shown = set()

for idx, s in enumerate(rep_scenarios):
    r, c = idx // ncols + 1, idx % ncols + 1
    sub = df[(df.scenario_num == s) & (df.label == nac_label) & (df.iteration == 1)]
    if sub.empty:
        continue
    tt = sub.iloc[0].get("turn_tokens", [])
    if not tt:
        continue
    x_turns = list(range(1, len(tt) + 1))
    for key, name, color in categories:
        show = name not in legend_shown
        if show:
            legend_shown.add(name)
        fig.add_trace(go.Bar(
            x=x_turns, y=[t.get(key, 0) for t in tt],
            name=name, marker_color=color, legendgroup=name,
            showlegend=show,
        ), row=r, col=c)
    fig.update_xaxes(dtick=1, row=r, col=c)

fig.update_yaxes(title_text="Tokens", row=1, col=1)
fig.update_yaxes(title_text="Tokens", row=2, col=1)
fig.update_layout(**LAYOUT, barmode="stack", height=600, width=1000,
    title=dict(text="Per-Turn Token Breakdown (Notion Agent CLI Sonnet, iteration 1)",
               y=0.98, x=0.5, xanchor="center", font_size=15),
    margin=dict(t=90, b=40, l=80, r=30),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center",
                font_size=11))
charts["bench-token-breakdown"] = fig
fig.show()

## 8. Turns vs Cost — Efficiency Frontier

Each dot is one session. Reveals the structural relationship: more turns = more cost (context replay). NAC clusters in the low-turn/low-cost corner; MCP spreads along the cost axis.

In [15]:
fig = go.Figure()

for label in labels:
    sub = df[df.label == label]
    color = PALETTE.get(label, "#888")

    fig.add_trace(go.Scatter(
        x=sub.num_turns, y=sub.cost_usd, mode="markers",
        name=label, marker=dict(color=color, size=10, opacity=0.8),
        hovertemplate="%{text}<br>Turns: %{x}<br>Cost: $%{y:.4f}<extra>" + label + "</extra>",
        text=sub.scenario,
    ))

    if len(sub) > 2:
        z = np.polyfit(sub.num_turns, sub.cost_usd, 2)
        p = np.poly1d(z)
        x_range = np.linspace(sub.num_turns.min(), sub.num_turns.max(), 50)
        fig.add_trace(go.Scatter(
            x=x_range, y=p(x_range), mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False, hoverinfo="skip",
        ))

fig.update_layout(**LAYOUT, height=500, width=850,
    title=dict(text="Turns vs Cost: Each Dot is One Session",
               y=0.97, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=80, b=60, l=70, r=30),
    yaxis_tickprefix="$", xaxis_title="Agentic Turns", yaxis_title="Cost (USD)",
    legend=LEGEND_TOP)
fig.show()
charts["bench-turns-vs-cost"] = fig


## 9. Output Token Ratio — I/O Format Impact

Compares total output tokens (what Claude generates) per scenario. MCP requires Claude to construct verbose JSON block structures; NAC accepts markdown, producing smaller output. This compounds via context replay on subsequent turns.

In [16]:
if "total_output" in df.columns and "total_input" in df.columns:
    io_agg = df.groupby(["scenario_num", "scenario_short", "label"]).agg(
        input_mean=("total_input", "mean"),
        output_mean=("total_output", "mean"),
    ).reset_index()
    io_agg["output_ratio"] = io_agg["output_mean"] / (io_agg["input_mean"] + io_agg["output_mean"]) * 100

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Total Output Tokens (avg)", "Output / Total Ratio"],
                        horizontal_spacing=0.14)

    for label in labels:
        sub = io_agg[io_agg.label == label].sort_values("scenario_num")
        color = PALETTE.get(label, "#888")

        fig.add_trace(go.Bar(
            name=label, x=sub.scenario_short, y=sub.output_mean,
            marker_color=color, opacity=0.85, legendgroup=label, showlegend=True,
            hovertemplate="%{x}<br>Output: %{y:,.0f} tokens<extra>" + label + "</extra>",
        ), row=1, col=1)

        fig.add_trace(go.Bar(
            name=label, x=sub.scenario_short, y=sub.output_ratio,
            marker_color=color, opacity=0.85, legendgroup=label, showlegend=False,
            hovertemplate="%{x}<br>Output ratio: %{y:.1f}%<extra>" + label + "</extra>",
        ), row=1, col=2)

    fig.update_yaxes(title_text="Tokens", row=1, col=1)
    fig.update_yaxes(title_text="%", row=1, col=2)
    fig.update_layout(**LAYOUT, barmode="group", height=520, width=1100,
        title=dict(text="Output Verbosity: I/O Format Impact", y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=120, b=50, l=60, r=30), legend=LEGEND_TOP)
    fig.show()
else:
    print("Token data not available")

## 9a. Outlier Sessions

In [17]:
# Outlier sessions: high cost (>2 SD from scenario mean), unusual turn count, or retries
outlier_rows = []
for s in sorted(df.scenario_num.unique()):
    for label in labels:
        sub = df[(df.scenario_num == s) & (df.label == label)]
        if len(sub) < 2:
            continue
        cost_mean, cost_std = sub.cost_usd.mean(), sub.cost_usd.std()
        turns_mean, turns_std = sub.num_turns.mean(), sub.num_turns.std()
        for _, row in sub.iterrows():
            reasons = []
            if cost_std > 0 and abs(row.cost_usd - cost_mean) > 2 * cost_std:
                reasons.append(f"cost {row.cost_usd:.4f} vs mean {cost_mean:.4f}")
            if turns_std > 0 and abs(row.num_turns - turns_mean) > 2 * turns_std:
                reasons.append(f"turns {row.num_turns} vs mean {turns_mean:.1f}")
            seq = row.get("action_sequence", [])
            if isinstance(seq, list) and len(seq) > len(set(seq)) + 1:
                reasons.append(f"retries in {' -> '.join(seq)}")
            if reasons:
                outlier_rows.append({
                    "Session": row["name"],
                    "Label": label,
                    "Turns": row.num_turns,
                    "Cost": f"${row.cost_usd:.4f}",
                    "Reason": "; ".join(reasons),
                })

if outlier_rows:
    outlier_df = pd.DataFrame(outlier_rows)
    display(Markdown(f"**{len(outlier_rows)} outlier sessions detected:**"))
    display(outlier_df)
else:
    display(Markdown("No outlier sessions detected (all within 2 SD)."))

**4 outlier sessions detected:**

,Session,Label,Turns,Cost,Reason
0,nac-s2-5,Notion Agent CLI (Opus),6,$0.1754,retries in queryDatabase -> createPage -> crea...
1,nac-s9-4,Notion Agent CLI (Opus),6,$0.2429,retries in createPage -> createPage -> createP...
2,nac-s10-3,Notion Agent CLI (Opus),5,$0.2113,retries in importTable -> importTable -> impor...
3,nac-s10-4,Notion Agent CLI (Opus),5,$0.2194,retries in importTable -> importTable -> impor...


## 9b. Cache Effectiveness

In [18]:
# Cache hit rate per scenario, split by model
if all(c in df.columns for c in ["total_cache_read", "total_input", "total_cache_creation"]):
    cdf = df.copy()
    cdf["total_ctx"] = cdf["total_input"] + cdf["total_cache_read"] + cdf["total_cache_creation"]
    cdf["cache_pct"] = cdf["total_cache_read"] / cdf["total_ctx"].replace(0, np.nan) * 100

    pairs = _model_pairs(labels)
    ncols = len(pairs) if pairs else 1
    fig = make_subplots(rows=1, cols=ncols,
        subplot_titles=[m for m, _, _ in pairs] if pairs else [""],
        horizontal_spacing=0.14, shared_yaxes=True)

    scenarios = sorted(cdf.scenario_num.unique())
    slabels = [f"S{s}" for s in scenarios]

    for col, (model, cli_l, mcp_l) in enumerate(pairs, 1):
        for label, color, name in [(cli_l, NAC_COLOR, "Notion Agent CLI"), (mcp_l, MCP_COLOR, "MCP")]:
            sub = cdf[cdf.label == label].groupby("scenario_num").cache_pct.mean().reindex(scenarios)
            fig.add_trace(go.Bar(
                x=slabels, y=sub.values, name=name, legendgroup=name,
                marker_color=color, opacity=0.85, showlegend=(col == 1),
            ), row=1, col=col)
        fig.update_xaxes(tickangle=-45, row=1, col=col)

    fig.update_yaxes(title_text="Cache hit rate (%)", range=[0, 105], row=1, col=1)
    fig.update_layout(**LAYOUT, barmode="group", height=420, width=500 * ncols + 150,
        title=dict(text="Cache Hit Rate by Scenario",
                   y=0.97, x=0.5, xanchor="center", font_size=15),
        margin=dict(t=80, b=70, l=70, r=30),
        legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.5, xanchor="center"))
    charts["bench-cache"] = fig
    fig.show()
else:
    print("Token columns not available for cache analysis")

## 10. Export to assets/

Re-run this cell to overwrite the PNGs used by README.md. Requires `kaleido` (`pip install kaleido`).

In [19]:
ASSETS_DIR.mkdir(exist_ok=True)
SCALE = 2
short_labels = {l: l.replace("Notion MCP", "MCP") for l in labels}

# ── Write PNGs from charts dict (populated by earlier cells) ──────────────────
for name, fig in charts.items():
    png_name = name if name.endswith(".png") else f"{name}.png"
    out = ASSETS_DIR / png_name
    fig.write_image(str(out), scale=SCALE)
    print(f"  -> {out}")
print(f"\nExported {len(charts)} charts to {ASSETS_DIR}")

# ── Export summary-stats.json ─────────────────────────────────────────────────
summary_stats = {"scenarios": {}, "models": {}}

for s in sorted(df.scenario_num.unique()):
    scenario_stats = {}
    for label in labels:
        sub = df[(df.scenario_num == s) & (df.label == label)]
        if sub.empty:
            continue
        lo, hi = ci_95(sub.cost_usd.values)
        scenario_stats[short_labels.get(label, label)] = {
            "cost_mean": round(float(sub.cost_usd.mean()), 6),
            "cost_std": round(float(sub.cost_usd.std()), 6),
            "cost_ci_lo": round(float(lo), 6) if not np.isnan(lo) else None,
            "cost_ci_hi": round(float(hi), 6) if not np.isnan(hi) else None,
            "turns_mean": round(float(sub.num_turns.mean()), 2),
            "n": int(len(sub)),
        }
    summary_stats["scenarios"][f"S{s}"] = scenario_stats

for model, cli_l, mcp_l in _model_pairs(labels):
    nac_all = df[df.label == cli_l]
    mcp_all = df[df.label == mcp_l]
    d = cohens_d(mcp_all.cost_usd.values, nac_all.cost_usd.values)
    summary_stats["models"][model or "default"] = {
        "nac_total_cost": round(float(nac_all.cost_usd.sum()), 4),
        "mcp_total_cost": round(float(mcp_all.cost_usd.sum()), 4),
        "nac_avg_turns": round(float(nac_all.num_turns.mean()), 2),
        "mcp_avg_turns": round(float(mcp_all.num_turns.mean()), 2),
        "saving_pct": round(float((1 - nac_all.cost_usd.sum() / mcp_all.cost_usd.sum()) * 100), 1)
            if mcp_all.cost_usd.sum() > 0 else 0,
        "cohens_d": round(float(d), 2) if not np.isnan(d) else None,
    }

stats_path = ASSETS_DIR / "summary-stats.json"
stats_path.write_text(json.dumps(summary_stats, indent=2) + "\n")
print(f"  -> {stats_path}")

  -> assets/bench-turns-cost.png
  -> assets/bench-turns-cost-paired.png
  -> assets/bench-savings.png
  -> assets/bench-cumulative.png
  -> assets/bench-token-breakdown.png
  -> assets/bench-turns-vs-cost.png
  -> assets/bench-cache.png

Exported 7 charts to assets
  -> assets/summary-stats.json
